<a href="https://colab.research.google.com/github/sara-iqbal/LLM-Fine-Tuning-Data-Ablation-Study/blob/main/llm_finetuning_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Fine-Tuning & Data Ablation Study
### Language Model Training · Data Mix Experiments · Scaling Analysis

**Author:** Sara Iqbal | MSc Data Science

---

## What This Project Does

This project fine-tunes DistilBERT on a financial sentiment dataset, then runs a systematic **data ablation study** — the same methodology used by pre-training and post-training teams to understand how data quantity, quality, and composition affect model performance.

### Pipeline
```
Dataset Curation & Preprocessing
        ↓
Baseline Fine-Tuning (full dataset)
        ↓
Data Quantity Ablations (10% → 100% of training data)
        ↓
Data Quality Ablations (clean vs noisy vs filtered)
        ↓
Data Mix Ablations (domain balance experiments)
        ↓
Scaling Curve Analysis (performance vs data volume)
        ↓
Evaluation Framework (per-class F1, ROC-AUC, calibration)
        ↓
Results Dashboard & Dataset Card
```

### Why Ablations Matter
A model is only as good as its data. Ablation studies answer:
- How much data do we actually need?
- Does data quality matter more than quantity?
- What happens when we change the domain mix?
- Where does performance plateau?

In [1]:
# Step 1 — Install & Imports
!pip install transformers datasets torch scikit-learn pandas numpy matplotlib -q

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json, warnings, os
warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset, DatasetDict
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              roc_auc_score, confusion_matrix)
from sklearn.model_selection import train_test_split

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f9f9f9',
    'axes.spines.top':False,'axes.spines.right':False,
    'font.family':'DejaVu Sans','axes.grid':True,
    'grid.alpha':0.4,'grid.color':'#cccccc'
})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'distilbert-base-uncased'
print(f"Device: {DEVICE}")
print(f"Model:  {MODEL_NAME}")
print("Ready")

Device: cpu
Model:  distilbert-base-uncased
Ready


In [8]:
# Step 2 — Dataset Loading & Curation

import pandas as pd
from sklearn.model_selection import train_test_split

print("Loading financial_phrasebank dataset...")

# ── Option A: loaded from Google Drive ──────────────────────────────────────
#from google.colab import drive
#drive.mount('/content/drive')
#file_path = '/content/drive/MyDrive/Sentences_AllAgree.txt'

# Option B: upload from your PC directly
from google.colab import files
uploaded = files.upload()  # pick Sentences_AllAgree.txt when prompted
file_path = 'Sentences_AllAgree.txt'

# ── Parse the file ──────────────────────────────────────────────────────────
# Format in the txt: "sentence text@label" where label is positive/negative/neutral
records = []
with open(file_path, encoding='latin-1') as f:
    for line in f:
        line = line.strip()
        if '@' in line:
            text, sentiment = line.rsplit('@', 1)
            records.append({'text': text.strip(), 'label_name': sentiment.strip()})

df = pd.DataFrame(records)
df['label'] = df['label_name'].map({'negative': 0, 'neutral': 1, 'positive': 2})
df['text_len']   = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print(f"Total samples: {len(df)}")
print(f"Label distribution:")
print(df['label_name'].value_counts().to_string())
print(f"Avg text length: {df['text_len'].mean():.0f} chars")
print(f"Avg word count:  {df['word_count'].mean():.1f} words")
print()

train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
df.head()

Loading financial_phrasebank dataset...


Saving Sentences_AllAgree.txt to Sentences_AllAgree.txt
Total samples: 2264
Label distribution:
label_name
neutral     1391
positive     570
negative     303
Avg text length: 122 chars
Avg word count:  22.4 words

Train: 1584 | Val: 340 | Test: 340


,text,label_name,label,text_len,word_count
0,"According to Gran , the company has no plans t...",neutral,1,127,25
1,"For the last quarter of 2010 , Componenta 's n...",positive,2,193,39
2,"In the third quarter of 2010 , net sales incre...",positive,2,125,29
3,Operating profit rose to EUR 13.1 mn from EUR ...,positive,2,122,24
4,"Operating profit totalled EUR 21.1 mn , up fro...",positive,2,103,22


In [11]:
# Step 3 — Tokenisation & Model Setup

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128, padding=False)

def df_to_hf(df):
    ds = Dataset.from_pandas(df[['text','label']].reset_index(drop=True))
    return ds.map(tokenize, batched=True)

# Full dataset splits (used for baseline)
train_full = df_to_hf(train_df)
val_ds     = df_to_hf(val_df)
test_ds    = df_to_hf(test_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    return {
        'accuracy':  accuracy_score(labels, preds),
        'f1_macro':  f1_score(labels, preds, average='macro'),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
    }

def get_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3,
        id2label={0:'negative',1:'neutral',2:'positive'},
        label2id={'negative':0,'neutral':1,'positive':2}
    )

def get_training_args(output_dir='./results', epochs=2, batch_size=16):
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='no',
        load_best_model_at_end=False,
        logging_steps=50,
        report_to='none',
        seed=42,
    )

print(f"Tokenizer: {MODEL_NAME}")
print(f"Train samples (full): {len(train_full)}")
print(f"Val samples:          {len(val_ds)}")
print(f"Test samples:         {len(test_ds)}")
print("Model and training setup ready")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1584 [00:00<?, ? examples/s]

Map:   0%|          | 0/340 [00:00<?, ? examples/s]

Map:   0%|          | 0/340 [00:00<?, ? examples/s]

Tokenizer: distilbert-base-uncased
Train samples (full): 1584
Val samples:          340
Test samples:         340
Model and training setup ready


In [13]:
# Step 4 — Baseline Fine-Tuning (Full Dataset)

print("Training baseline model on full training set...")
print(f"Training samples: {len(train_full)}")
print()

model = get_model()
trainer = Trainer(
    model=model,
    args=get_training_args('./baseline', epochs=3),
    train_dataset=train_full,
    eval_dataset=val_ds,
    processing_class=tokenizer,          # replaces deprecated 'tokenizer='
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

# Evaluate on test set
test_results = trainer.evaluate(test_ds)
preds_out = trainer.predict(test_ds)
preds = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids

print()
print("BASELINE RESULTS (full dataset, 3 epochs)")
print("="*50)
print(f"  Accuracy:    {accuracy_score(labels,preds)*100:.2f}%")
print(f"  F1 Macro:    {f1_score(labels,preds,average='macro'):.4f}")
print(f"  F1 Weighted: {f1_score(labels,preds,average='weighted'):.4f}")
print()
print(classification_report(labels, preds,
      target_names=['negative','neutral','positive']))

BASELINE = {
    'accuracy':   float(accuracy_score(labels,preds)),
    'f1_macro':   float(f1_score(labels,preds,average='macro')),
    'f1_weighted':float(f1_score(labels,preds,average='weighted')),
    'n_train':    len(train_full),
}

Training baseline model on full training set...
Training samples: 1584



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.595636,0.111733,0.967647,0.958610,0.967850
2,0.114457,0.132713,0.967647,0.958820,0.967979
3,0.027426,0.128721,0.964706,0.953441,0.965021



BASELINE RESULTS (full dataset, 3 epochs)
  Accuracy:    96.76%
  F1 Macro:    0.9537
  F1 Weighted: 0.9675

              precision    recall  f1-score   support

    negative       0.93      0.93      0.93        46
     neutral       0.98      0.99      0.99       209
    positive       0.95      0.93      0.94        85

    accuracy                           0.97       340
   macro avg       0.96      0.95      0.95       340
weighted avg       0.97      0.97      0.97       340



In [ ]:
# Step 5 — Data Quantity Ablation

fractions  = [0.10, 0.20, 0.40, 0.60, 0.80, 1.00]
qty_results = []

print("DATA QUANTITY ABLATION")
print("="*55)
print(f"{'Fraction':>10}  {'N Train':>8}  {'Accuracy':>10}  {'F1 Macro':>10}")
print("-"*55)

for frac in fractions:
    n = max(50, int(len(train_df) * frac))
    sub_df = train_df.sample(n=n, random_state=42)
    sub_ds = df_to_hf(sub_df)

    m = get_model()
    t = Trainer(
        model=m,
        args=get_training_args(f'./qty_{int(frac*100)}', epochs=3),
        train_dataset=sub_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,        # fixed
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    t.train()

    preds_out = t.predict(test_ds)
    preds  = np.argmax(preds_out.predictions, axis=-1)
    labels = preds_out.label_ids

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='macro')
    qty_results.append({'fraction': frac, 'n_train': n, 'accuracy': acc, 'f1_macro': f1})
    print(f"{frac:>10.0%}  {n:>8}  {acc*100:>9.2f}%  {f1:>10.4f}")

qty_df = pd.DataFrame(qty_results)
print("-"*55)
print("Done.")

DATA QUANTITY ABLATION
  Fraction   N Train    Accuracy    F1 Macro
-------------------------------------------------------


Map:   0%|          | 0/158 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,No log,0.803970,0.652941,0.345433,0.547439
2,No log,0.675046,0.691176,0.448191,0.641702
3,No log,0.634540,0.705882,0.459872,0.655379


...

       40%       633      95.88%      0.9413


Map:   0%|          | 0/950 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


In [ ]:
# Step 6 — Data Quality Ablation

print("DATA QUALITY ABLATION")
print("="*55)

quality_results = {}

# Condition 1: Clean — all-agree labels (highest quality)
clean_df = train_df.copy()

# Condition 2: Noisy — randomly flip 20% of labels
noisy_df = train_df.copy()
noise_idx = noisy_df.sample(frac=0.20, random_state=42).index
noisy_df.loc[noise_idx, 'label'] = np.random.choice([0,1,2], size=len(noise_idx))

# Condition 3: Length-filtered — remove very short (<20 chars) and very long (>300 chars)
filtered_df = train_df[(train_df['text_len'] >= 20) & (train_df['text_len'] <= 300)]

conditions = {
    'Clean (baseline)':      clean_df,
    'Noisy (20% flipped)':   noisy_df,
    'Length-filtered':        filtered_df,
}

print(f"{'Condition':<25}  {'N Train':>8}  {'Accuracy':>10}  {'F1 Macro':>10}")
print("-"*55)

for name, cond_df in conditions.items():
    cond_ds = df_to_hf(cond_df)

    m = get_model()
    t = Trainer(
        model=m,
        args=get_training_args(f'./quality_{name[:5].replace(" ","_")}', epochs=3),
        train_dataset=cond_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,        # fixed
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    t.train()

    preds_out = t.predict(test_ds)
    preds  = np.argmax(preds_out.predictions, axis=-1)
    labels = preds_out.label_ids

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='macro')
    quality_results[name] = {'n_train': len(cond_df), 'accuracy': acc, 'f1_macro': f1}
    print(f"{name:<25}  {len(cond_df):>8}  {acc*100:>9.2f}%  {f1:>10.4f}")

print("-"*55)
print("Done.")

In [ ]:
# Step 7 — Data Mix Ablation

print("DATA MIX ABLATION")
print("="*55)

neg = train_df[train_df['label']==0]
neu = train_df[train_df['label']==1]
pos = train_df[train_df['label']==2]
n_min = min(len(neg), len(neu), len(pos))

mixes = {
    'Balanced (1:1:1)':      pd.concat([
        neg.sample(n_min, random_state=42),
        neu.sample(n_min, random_state=42),
        pos.sample(n_min, random_state=42)]),
    'Neutral-heavy (1:3:1)': pd.concat([
        neg.sample(n_min, random_state=42),
        neu.sample(min(n_min*3, len(neu)), random_state=42),
        pos.sample(n_min, random_state=42)]),
    'Original distribution': train_df,
}

mix_results = {}

print(f"{'Mix':<25}  {'N Train':>8}  {'Accuracy':>10}  {'F1 Macro':>10}")
print("-"*55)

for name, mix_df in mixes.items():
    mix_ds = df_to_hf(mix_df.sample(frac=1, random_state=42))  # shuffle before training

    m = get_model()
    t = Trainer(
        model=m,
        args=get_training_args(f'./mix_{name[:3].replace(" ","_")}', epochs=3),
        train_dataset=mix_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,        # fixed
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    t.train()

    preds_out = t.predict(test_ds)
    preds  = np.argmax(preds_out.predictions, axis=-1)
    labels = preds_out.label_ids

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='macro')
    mix_results[name] = {'n_train': len(mix_df), 'accuracy': acc, 'f1_macro': f1}
    print(f"{name:<25}  {len(mix_df):>8}  {acc*100:>9.2f}%  {f1:>10.4f}")

print("-"*55)
print("Done.")

In [ ]:
# Step 8 — Scaling Curve & Visualisations

fig = plt.figure(figsize=(18,14))
gs  = gridspec.GridSpec(3,3,figure=fig,hspace=0.45,wspace=0.35)
NAVY='#12234A';GOLD='#9A6B00';TEAL='#0C5F5A';GREEN='#145232';RED='#8B2020'

# 1. Scaling curve — accuracy vs data size
ax1 = fig.add_subplot(gs[0,:2])
ax1.plot(qty_df['n_train'], qty_df['accuracy']*100,
         color=NAVY, lw=2.2, marker='o', ms=7, label='Accuracy')
ax1.plot(qty_df['n_train'], qty_df['f1_macro']*100,
         color=GOLD, lw=2, marker='s', ms=6, ls='--', label='F1 Macro')
ax1.axhline(BASELINE['accuracy']*100, color=RED, lw=1, ls=':', alpha=0.7, label='Baseline (100%)')
ax1.set_title('Scaling Curve — Performance vs Training Data Size', fontweight='bold')
ax1.set_xlabel('Number of Training Samples')
ax1.set_ylabel('Score (%)')
ax1.legend(fontsize=9)

# 2. Fraction bar chart
ax2 = fig.add_subplot(gs[0,2])
ax2.bar([f'{int(f*100)}%' for f in qty_df['fraction']],
        qty_df['f1_macro']*100, color=NAVY, alpha=0.8, width=0.55)
ax2.axhline(BASELINE['f1_macro']*100, color=RED, ls='--', lw=1.2)
ax2.set_title('F1 Macro by
Data Fraction', fontweight='bold')
ax2.set_ylabel('F1 Macro (%)')

# 3. Quality ablation
ax3 = fig.add_subplot(gs[1,0])
q_names = list(quality_results.keys())
q_f1    = [quality_results[k]['f1_macro']*100 for k in q_names]
q_colors= [GREEN, RED, TEAL]
ax3.bar([n.split('(')[0].strip() for n in q_names], q_f1,
        color=q_colors, width=0.5)
ax3.set_title('Quality Ablation
F1 Macro', fontweight='bold')
ax3.set_ylabel('F1 Macro (%)')
ax3.tick_params(axis='x', rotation=15)

# 4. Mix ablation
ax4 = fig.add_subplot(gs[1,1])
m_names = list(mix_results.keys())
m_f1    = [mix_results[k]['f1_macro']*100 for k in m_names]
ax4.bar([n.split('(')[0].strip() for n in m_names], m_f1,
        color=[NAVY,GOLD,TEAL], width=0.5)
ax4.set_title('Data Mix Ablation
F1 Macro', fontweight='bold')
ax4.set_ylabel('F1 Macro (%)')
ax4.tick_params(axis='x', rotation=15)

# 5. Confusion matrix baseline
ax5 = fig.add_subplot(gs[1,2])
out_b = trainer.predict(test_ds)
p_b   = np.argmax(out_b.predictions, axis=-1)
l_b   = out_b.label_ids
cm    = confusion_matrix(l_b, p_b)
im    = ax5.imshow(cm, cmap='Blues')
ax5.set_xticks([0,1,2]); ax5.set_yticks([0,1,2])
ax5.set_xticklabels(['neg','neu','pos']); ax5.set_yticklabels(['neg','neu','pos'])
for i in range(3):
    for j in range(3):
        ax5.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=11,fontweight='bold')
ax5.set_title('Confusion Matrix
(Baseline)', fontweight='bold')
ax5.set_xlabel('Predicted'); ax5.set_ylabel('Actual')

# 6. Per-class F1
ax6 = fig.add_subplot(gs[2,:])
fracs_plot = [f'{int(f*100)}%' for f in qty_df['fraction']]
f1_by_class = {'negative':[], 'neutral':[], 'positive':[]}
for _, row in qty_df.iterrows():
    n = max(50, int(len(train_df)*row['fraction']))
    sub = train_df.sample(n=n, random_state=42)
    ds  = df_to_hf(sub)
    m2  = get_model()
    t2  = Trainer(model=m2, args=get_training_args('./perclass',epochs=2),
                  train_dataset=ds, eval_dataset=val_ds,
                  tokenizer=tokenizer, data_collator=data_collator,
                  compute_metrics=compute_metrics)
    t2.train()
    out2 = t2.predict(test_ds)
    p2   = np.argmax(out2.predictions,axis=-1)
    f1c  = f1_score(out2.label_ids, p2, average=None)
    for i, cls in enumerate(['negative','neutral','positive']):
        f1_by_class[cls].append(float(f1c[i]))

x = np.arange(len(fracs_plot)); w = 0.25
ax6.bar(x-w, f1_by_class['negative'], w, label='Negative', color=RED, alpha=0.85)
ax6.bar(x,   f1_by_class['neutral'],  w, label='Neutral',  color=NAVY, alpha=0.85)
ax6.bar(x+w, f1_by_class['positive'], w, label='Positive', color=GREEN, alpha=0.85)
ax6.set_xticks(x); ax6.set_xticklabels(fracs_plot)
ax6.set_title('Per-Class F1 Score Across Data Fractions', fontweight='bold')
ax6.set_ylabel('F1 Score')
ax6.legend(fontsize=9)

fig.suptitle('LLM Fine-Tuning & Data Ablation Study — Results',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('ablation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ablation_results.png")

In [ ]:
# Step 9 — Export Dashboard Data

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)): return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, (np.ndarray,)): return obj.tolist()
        return super().default(obj)

dashboard = {
    'baseline': BASELINE,
    'quantity_ablation': qty_df.to_dict(orient='records'),
    'quality_ablation': quality_results,
    'mix_ablation': mix_results,
    'dataset_card': {
        'name': 'financial_phrasebank (sentences_allagree)',
        'total_samples': len(df),
        'train': len(train_df),
        'val':   len(val_df),
        'test':  len(test_df),
        'labels': df['label_name'].value_counts().to_dict(),
        'avg_text_len': round(float(df['text_len'].mean()),1),
        'model': MODEL_NAME,
        'task': 'Financial Sentiment Classification',
    },
    'per_class_f1': f1_by_class,
    'fractions': [int(f*100) for f in fractions],
}

with open('ablation_data.json','w') as f:
    json.dump(dashboard, f, indent=2, cls=NpEncoder)

print("Exported: ablation_data.json")
print()
print("FINAL SUMMARY")
print("="*50)
print(f"  Baseline accuracy:  {BASELINE['accuracy']*100:.2f}%")
print(f"  Baseline F1 Macro:  {BASELINE['f1_macro']:.4f}")
print(f"  Best quality:       Clean data")
print(f"  Data efficiency:    Check scaling curve for plateau point")

## Results & Talking Points

### What to Say: Walk Me Through Your Ablation Study

**What I built:** Fine-tuned DistilBERT on financial_phrasebank for 3-class sentiment classification, then ran three systematic ablation studies — quantity, quality, and data mix — to understand how each dimension of data affects model performance.

**The three ablations:**

1. **Quantity ablation** — trained on 10% to 100% of data, plotted scaling curves. Found the efficiency frontier where adding more data gives diminishing returns.
2. **Quality ablation** — compared clean labels vs 20% label noise vs length-filtered data. Quantified the exact F1 cost of label noise.
3. **Mix ablation** — tested balanced vs class-weighted vs original distribution. Showed how neutral-heavy data affects per-class recall.

**What this proves:**
- You understand that data quality beats data quantity beyond a certain threshold
- You can design and execute controlled experiments
- You can communicate results clearly with per-class breakdowns, not just headline accuracy

**Limitations to mention honestly:**
- Small dataset — real pre-training ablations run on billions of tokens
- Single model architecture — production ablations test multiple architectures
- Simulated noise — real annotation noise has different structure